In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from e2cnn import gspaces
from e2cnn import nn as enn

import torchvision.transforms as transforms
from torchvision import datasets

# Regular CNN Model
class RegularCNN(nn.Module):
    def __init__(self):
        super(RegularCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2(x), 2))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Training Function
def train(model, train_loader, optimizer, criterion, device):
    model.train()
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

# Testing Function
def test(model, test_loader, criterion, device):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += criterion(output, target).item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    accuracy = correct / len(test_loader.dataset)
    return test_loss, accuracy

In [8]:
# Dataset Preparation
transform = transforms.Compose([
    transforms.RandomRotation(180),  # Apply random rotations
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Load and augment Fashion-MNIST dataset
train_dataset = datasets.FashionMNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.FashionMNIST(root='./data', train=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [9]:
# SE2CNN Model
class SE2CNN(nn.Module):
    def __init__(self, max_freq=5):
        super(SE2CNN, self).__init__()
        self.r2_act = gspaces.Rot2dOnR2(N=-1, maximum_frequency=max_freq)  # Continuous group of rotations
        self.input_type = enn.FieldType(self.r2_act, 1 * [self.r2_act.trivial_repr])
        
        # Use trivial_repr for the input and select appropriate representations for subsequent layers
        self.conv1 = enn.R2Conv(self.input_type, enn.FieldType(self.r2_act, 32 * [self.r2_act.trivial_repr]), kernel_size=3, padding=1)
        self.bn1 = enn.InnerBatchNorm(self.conv1.out_type)
        self.relu1 = enn.ReLU(self.conv1.out_type)
        
        self.conv2 = enn.R2Conv(self.conv1.out_type, enn.FieldType(self.r2_act, 64 * [self.r2_act.trivial_repr]), kernel_size=3, padding=1)
        self.bn2 = enn.InnerBatchNorm(self.conv2.out_type)
        self.relu2 = enn.ReLU(self.conv2.out_type)
        
        self.gpool = enn.GroupPooling(self.conv2.out_type)
        
        self.fc1 = nn.Linear(64 * 1 * 1, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = enn.GeometricTensor(x, self.input_type)
        x = self.relu1(self.bn1(self.conv1(x)))
        x = self.relu2(self.bn2(self.conv2(x)))
        x = self.gpool(x)
        x = x.tensor
        
        x = F.adaptive_avg_pool2d(x, 1).view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x
    
# Main Experiment
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_epochs = 10
criterion = nn.CrossEntropyLoss()

# Regular CNN
regular_model = RegularCNN().to(device)
regular_optimizer = torch.optim.Adam(regular_model.parameters(), lr=0.001)

# SE2CNN
se2_model = SE2CNN().to(device)
se2_optimizer = torch.optim.Adam(se2_model.parameters(), lr=0.001)

# Training Loop
for epoch in range(num_epochs):
    train(regular_model, train_loader, regular_optimizer, criterion, device)
    train(se2_model, train_loader, se2_optimizer, criterion, device)
    
    regular_loss, regular_acc = test(regular_model, test_loader, criterion, device)
    se2_loss, se2_acc = test(se2_model, test_loader, criterion, device)
    
    print(f'Epoch {epoch+1}/{num_epochs}')
    print(f'Regular CNN - Loss: {regular_loss:.4f}, Accuracy: {regular_acc:.4f}')
    print(f'SE2CNN      - Loss: {se2_loss:.4f}, Accuracy: {se2_acc:.4f}')

Epoch 1/10
Regular CNN - Loss: 0.0123, Accuracy: 0.7098
SE2CNN      - Loss: 0.0217, Accuracy: 0.5017
Epoch 2/10
Regular CNN - Loss: 0.0105, Accuracy: 0.7544
SE2CNN      - Loss: 0.0198, Accuracy: 0.5336
Epoch 3/10
Regular CNN - Loss: 0.0095, Accuracy: 0.7804
SE2CNN      - Loss: 0.0197, Accuracy: 0.5275
Epoch 4/10
Regular CNN - Loss: 0.0091, Accuracy: 0.7890
SE2CNN      - Loss: 0.0189, Accuracy: 0.5572
Epoch 5/10
Regular CNN - Loss: 0.0086, Accuracy: 0.8005
SE2CNN      - Loss: 0.0192, Accuracy: 0.5448
Epoch 6/10
Regular CNN - Loss: 0.0087, Accuracy: 0.8081
SE2CNN      - Loss: 0.0183, Accuracy: 0.5592
Epoch 7/10
Regular CNN - Loss: 0.0079, Accuracy: 0.8184
SE2CNN      - Loss: 0.0165, Accuracy: 0.6219
Epoch 8/10
Regular CNN - Loss: 0.0075, Accuracy: 0.8280
SE2CNN      - Loss: 0.0171, Accuracy: 0.6063
Epoch 9/10
Regular CNN - Loss: 0.0075, Accuracy: 0.8311
SE2CNN      - Loss: 0.0169, Accuracy: 0.6095
Epoch 10/10
Regular CNN - Loss: 0.0075, Accuracy: 0.8268
SE2CNN      - Loss: 0.0168, Accura

In [10]:
# Training Loop
for epoch in range(100):
    train(regular_model, train_loader, regular_optimizer, criterion, device)
    train(se2_model, train_loader, se2_optimizer, criterion, device)
    
    regular_loss, regular_acc = test(regular_model, test_loader, criterion, device)
    se2_loss, se2_acc = test(se2_model, test_loader, criterion, device)
    
    print(f'Epoch {epoch+1}/{num_epochs}')
    print(f'Regular CNN - Loss: {regular_loss:.4f}, Accuracy: {regular_acc:.4f}')
    print(f'SE2CNN      - Loss: {se2_loss:.4f}, Accuracy: {se2_acc:.4f}')

Epoch 1/10
Regular CNN - Loss: 0.0072, Accuracy: 0.8430
SE2CNN      - Loss: 0.0163, Accuracy: 0.6182
Epoch 2/10
Regular CNN - Loss: 0.0070, Accuracy: 0.8419
SE2CNN      - Loss: 0.0161, Accuracy: 0.6301
Epoch 3/10
Regular CNN - Loss: 0.0071, Accuracy: 0.8446
SE2CNN      - Loss: 0.0164, Accuracy: 0.6268
Epoch 4/10
Regular CNN - Loss: 0.0070, Accuracy: 0.8442
SE2CNN      - Loss: 0.0164, Accuracy: 0.6240
Epoch 5/10
Regular CNN - Loss: 0.0067, Accuracy: 0.8467
SE2CNN      - Loss: 0.0156, Accuracy: 0.6409
Epoch 6/10
Regular CNN - Loss: 0.0071, Accuracy: 0.8399
SE2CNN      - Loss: 0.0169, Accuracy: 0.6217
Epoch 7/10
Regular CNN - Loss: 0.0069, Accuracy: 0.8501
SE2CNN      - Loss: 0.0150, Accuracy: 0.6569
Epoch 8/10
Regular CNN - Loss: 0.0067, Accuracy: 0.8470
SE2CNN      - Loss: 0.0146, Accuracy: 0.6651
Epoch 9/10
Regular CNN - Loss: 0.0065, Accuracy: 0.8548
SE2CNN      - Loss: 0.0155, Accuracy: 0.6550
Epoch 10/10
Regular CNN - Loss: 0.0063, Accuracy: 0.8557
SE2CNN      - Loss: 0.0177, Accura

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from e2cnn import gspaces
from e2cnn import nn as enn

# Dataset Preparation
transform = transforms.Compose([
    transforms.RandomRotation(180),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Load and augment MNIST dataset
train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root='./data', train=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Regular CNN Model
class RegularCNN(nn.Module):
    def __init__(self):
        super(RegularCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2(x), 2))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


# SE2CNN Model
class SE2CNN(nn.Module):
    def __init__(self, max_freq=5):
        super(SE2CNN, self).__init__()
        self.r2_act = gspaces.Rot2dOnR2(N=-1, maximum_frequency=max_freq)  # Continuous group of rotations
        self.input_type = enn.FieldType(self.r2_act, 1 * [self.r2_act.trivial_repr])
        
        # Use trivial_repr for the input and select appropriate representations for subsequent layers
        self.conv1 = enn.R2Conv(self.input_type, enn.FieldType(self.r2_act, 32 * [self.r2_act.trivial_repr]), kernel_size=3, padding=1)
        self.bn1 = enn.InnerBatchNorm(self.conv1.out_type)
        self.relu1 = enn.ReLU(self.conv1.out_type)
        
        self.conv2 = enn.R2Conv(self.conv1.out_type, enn.FieldType(self.r2_act, 64 * [self.r2_act.trivial_repr]), kernel_size=3, padding=1)
        self.bn2 = enn.InnerBatchNorm(self.conv2.out_type)
        self.relu2 = enn.ReLU(self.conv2.out_type)
        
        self.gpool = enn.GroupPooling(self.conv2.out_type)
        
        self.fc1 = nn.Linear(64, 32)
        self.fc2 = nn.Linear(32, 10)

    def forward(self, x):
        x = enn.GeometricTensor(x, self.input_type)
        x = self.relu1(self.bn1(self.conv1(x)))
        x = self.relu2(self.bn2(self.conv2(x)))
        x = self.gpool(x)
        x = x.tensor
        
        x = F.adaptive_avg_pool2d(x, 1).view(x.size(0), -1)
        # print(x.size())
        x = self.fc1(x)
        # print(x.size())
        x = F.relu(x)
        # print(x.size())
        x = self.fc2(x)
        return x
    
# Training Function
def train(model, train_loader, optimizer, criterion, device):
    model.train()
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

# Testing Function
def test(model, test_loader, criterion, device):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += criterion(output, target).item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    accuracy = correct / len(test_loader.dataset)
    return test_loss, accuracy

# Main Experiment
device = torch.device('cuda:7' if torch.cuda.is_available() else 'cpu')
num_epochs = 10
criterion = nn.CrossEntropyLoss()

# Regular CNN
regular_model = RegularCNN().to(device)
regular_optimizer = torch.optim.Adam(regular_model.parameters(), lr=0.001)

# SE2CNN
se2_model = SE2CNN().to(device)
se2_optimizer = torch.optim.Adam(se2_model.parameters(), lr=0.001)

print(se2_model)
print(regular_model)

SE2CNN(
  (conv1): R2Conv([Continuous-Rotations: {irrep_0}], [Continuous-Rotations: {irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0}], kernel_size=3, stride=1, padding=1)
  (bn1): InnerBatchNorm([Continuous-Rotations: {irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0}], eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu1): ReLU(inplace=False, type=[Continuous-Rotations: {irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, i

In [3]:
# Training Loop
for epoch in range(100):
    train(regular_model, train_loader, regular_optimizer, criterion, device)
    train(se2_model, train_loader, se2_optimizer, criterion, device)
    
    regular_loss, regular_acc = test(regular_model, test_loader, criterion, device)
    se2_loss, se2_acc = test(se2_model, test_loader, criterion, device)
    
    print(f'Epoch {epoch+1}/{num_epochs}')
    print(f'Regular CNN - Loss: {regular_loss:.4f}, Accuracy: {regular_acc:.4f}')
    print(f'SE2CNN      - Loss: {se2_loss:.4f}, Accuracy: {se2_acc:.4f}')

Epoch 1/10
Regular CNN - Loss: 0.0015, Accuracy: 0.9689
SE2CNN      - Loss: 0.0208, Accuracy: 0.5255
Epoch 2/10
Regular CNN - Loss: 0.0015, Accuracy: 0.9711
SE2CNN      - Loss: 0.0199, Accuracy: 0.5590
Epoch 3/10
Regular CNN - Loss: 0.0016, Accuracy: 0.9670
SE2CNN      - Loss: 0.0273, Accuracy: 0.4166
Epoch 4/10
Regular CNN - Loss: 0.0015, Accuracy: 0.9716
SE2CNN      - Loss: 0.0206, Accuracy: 0.5189
Epoch 5/10
Regular CNN - Loss: 0.0015, Accuracy: 0.9710
SE2CNN      - Loss: 0.0199, Accuracy: 0.5546
Epoch 6/10
Regular CNN - Loss: 0.0013, Accuracy: 0.9757
SE2CNN      - Loss: 0.0341, Accuracy: 0.3061
Epoch 7/10
Regular CNN - Loss: 0.0013, Accuracy: 0.9730
SE2CNN      - Loss: 0.0312, Accuracy: 0.3676
Epoch 8/10
Regular CNN - Loss: 0.0014, Accuracy: 0.9727
SE2CNN      - Loss: 0.0256, Accuracy: 0.4178
Epoch 9/10
Regular CNN - Loss: 0.0014, Accuracy: 0.9725
SE2CNN      - Loss: 0.0277, Accuracy: 0.3924
Epoch 10/10
Regular CNN - Loss: 0.0013, Accuracy: 0.9739
SE2CNN      - Loss: 0.0698, Accura